# 原 Stage1 0bp 预测→目标→风险上限→账户损失诊断

完整自有状态 Stage1 与 SOXX 原生账户只读，固定 Tiered 费用。四年、十六季度、819 个连续 126 日窗口全部保留。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_original_0bp_signal_loss_chain_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][x]==0 for x in ('new_market_requests','new_model_fits','new_native_accounts','new_bootstraps'))
a=root/src['stage1_card'];b=root/src['component_card'];sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sha(a/'summary.json')==src['stage1_summary_sha256'] and sha(b/'summary.json')==src['component_summary_sha256']
assert sa['recorder_id']==src['stage1_recorder_id'] and sb['recorder_id']==src['component_recorder_id']
assert sa['nas_evidence']['sha256']==src['stage1_NAS_archive_sha256'] and sb['nas_evidence']['sha256']==src['component_NAS_archive_sha256']
for card_root,eid,rid,items in [(a,src['stage1_experiment_id'],src['stage1_recorder_id'],[('inputs/frame','stage1_frame_sha256'),('paths/A_active_research_replay_0','stage1_path_sha256')]),(b,src['component_experiment_id'],src['component_recorder_id'],[('analysis/daily','component_daily_sha256'),('analysis/periods','component_periods_sha256'),('analysis/rolling126','component_rolling_sha256')])]:
 for name,key in items:assert sha(card_root/'qlib_recorder'/eid/rid/'artifacts'/name)==src[key]
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['provider_uri']),region='us',kernels=1)
with R.uri_context(str(a/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=src['stage1_recorder_id'],experiment_id=src['stage1_experiment_id']);frame=rec.load_object('inputs/frame');path=rec.load_object('paths/A_active_research_replay_0')['replay']
with R.uri_context(str(b/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=src['component_recorder_id'],experiment_id=src['component_experiment_id']);components=rec.load_object('analysis/daily');source_periods=rec.load_object('analysis/periods');source_rolling=rec.load_object('analysis/rolling126')
assert len(frame)==len(path)==len(components)==944 and len(source_periods)==21 and len(source_rolling)==819
for name in ('signal_date','trade_date','exit_date'):
 x=pd.DatetimeIndex(pd.to_datetime(frame[name]));assert x.equals(pd.DatetimeIndex(pd.to_datetime(path[name]))) and x.equals(pd.DatetimeIndex(pd.to_datetime(components[name])))
assert (pd.to_datetime(frame.feature_available_at)<pd.to_datetime(frame.decision_at)).all()
calendar=pd.DatetimeIndex(D.calendar(start_time=str(frame.signal_date.iloc[0]),end_time=str(frame.signal_date.iloc[-1]),freq='day'))
assert len(calendar)==944 and calendar.equals(pd.DatetimeIndex(pd.to_datetime(frame.signal_date)))
assert np.max(np.abs(components[['cash_opportunity_log','SOXL_selection_log','Stage1_direct_fees_log','timing_and_benchmark_residual_log']].sum(axis=1)-components.paired_log_excess))<1e-12
assert abs(float(np.exp(components.paired_log_excess.sum()))-0.6860064369263443)<1e-9
raw_beta=3*frame.soxl_weight.astype(float)+frame.soxx_weight.astype(float);cap=path.current_beta_cap.astype(float);ending_beta=3*path.ending_soxl_weight.astype(float)+path.ending_soxx_weight.astype(float)
assert np.isfinite(raw_beta).all() and np.isfinite(cap).all() and np.isfinite(ending_beta).all()
d=pd.DataFrame({'signal_date':pd.to_datetime(frame.signal_date),'trade_date':pd.to_datetime(frame.trade_date),'exit_date':pd.to_datetime(frame.exit_date),'alpha_probability':frame.alpha_probability.astype(float),'tail_probability':frame.tail_probability.astype(float),'raw_nominal_beta':raw_beta,'own_state_beta_cap':cap,'cap_binding':raw_beta>cap+1e-8,'ending_beta':ending_beta,'open_execute':path.execute_open_target.astype(bool),'risk_state':path.risk_state.astype(int),'ending_cash_weight':path.ending_cash_weight.astype(float),'paired_log_excess':components.paired_log_excess.astype(float),'cash_opportunity_log':components.cash_opportunity_log.astype(float),'SOXL_selection_log':components.SOXL_selection_log.astype(float),'Stage1_direct_fees_log':components.Stage1_direct_fees_log.astype(float),'timing_and_benchmark_residual_log':components.timing_and_benchmark_residual_log.astype(float)})
assert d.notna().all().all() and d.signal_date.is_monotonic_increasing
static=pd.concat({'feature':d[['alpha_probability','tail_probability','raw_nominal_beta','own_state_beta_cap']],'label':d[['cash_opportunity_log','SOXL_selection_log','paired_log_excess']],'meta':d[['cap_binding','open_execute','ending_beta','ending_cash_weight']]},axis=1)
static.index=pd.MultiIndex.from_arrays([d.signal_date,['SOXL_SOXX_CASH']*len(d)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'diagnostic':(str(d.signal_date.iloc[0].date()),str(d.signal_date.iloc[-1].date()))})
assert len(dataset.prepare('diagnostic',col_set=['feature','label','meta']))==944
fields=['paired_log_excess','cash_opportunity_log','SOXL_selection_log','Stage1_direct_fees_log','timing_and_benchmark_residual_log']
def describe(g,kind,period,partial=False):
 return {'kind':kind,'period':str(period),'partial_boundary_period':bool(partial),'sessions':len(g),'first_trade':str(g.trade_date.iloc[0].date()),'last_exit':str(g.exit_date.iloc[-1].date()),'relative_log_excess':float(g.paired_log_excess.sum()),'relative_wealth':float(np.exp(g.paired_log_excess.sum())),**{x:float(g[x].sum()) for x in fields[1:]},'mean_alpha_probability':float(g.alpha_probability.mean()),'mean_tail_probability':float(g.tail_probability.mean()),'mean_raw_nominal_beta':float(g.raw_nominal_beta.mean()),'mean_own_state_beta_cap':float(g.own_state_beta_cap.mean()),'mean_ending_beta':float(g.ending_beta.mean()),'cap_binding_fraction':float(g.cap_binding.mean()),'open_execute_fraction':float(g.open_execute.mean()),'mean_ending_cash_weight':float(g.ending_cash_weight.mean()),'cap_binding_cash_log':float(g.loc[g.cap_binding,'cash_opportunity_log'].sum()),'nonbinding_cash_log':float(g.loc[~g.cap_binding,'cash_opportunity_log'].sum()),'cap_binding_selection_log':float(g.loc[g.cap_binding,'SOXL_selection_log'].sum()),'nonbinding_selection_log':float(g.loc[~g.cap_binding,'SOXL_selection_log'].sum())}
rows=[describe(d,'all','all')]
for freq,kind in [('Y','year'),('Q','quarter')]:
 for period,g in d.groupby(d.trade_date.dt.to_period(freq),sort=True):rows.append(describe(g,kind,period,period==d.trade_date.iloc[0].to_period(freq) or period==d.trade_date.iloc[-1].to_period(freq)))
periods=pd.DataFrame(rows);rolling=pd.DataFrame([describe(d.iloc[i:i+126],'rolling126',d.trade_date.iloc[i]) for i in range(len(d)-125)])
assert len(periods)==21 and len(rolling)==819
for target,source in ((periods,source_periods),(rolling,source_rolling)):
 assert len(target)==len(source) and target[['kind','period','sessions','first_trade','last_exit']].astype(str).equals(source[['kind','period','sessions','first_trade','last_exit']].astype(str))
 for col in ['relative_log_excess']+fields[1:]:np.testing.assert_allclose(target[col],source[col],atol=1e-10,rtol=0)
assert np.max(np.abs(periods[fields[1:]].sum(axis=1)-periods.relative_log_excess))<1e-10
quarter=periods.loc[periods.kind.eq('quarter')].set_index('period');key_quarters={k:quarter.loc[k].to_dict() for k in ('2024Q3','2026Q2','2026Q3')}
worst=rolling.loc[rolling.relative_log_excess.idxmin()].to_dict();best=rolling.loc[rolling.relative_log_excess.idxmax()].to_dict()
report={'status':'read_only_original_Stage1_signal_cap_loss_chain_closed_descriptive','source_Stage1_recorder_id':src['stage1_recorder_id'],'source_component_recorder_id':src['component_recorder_id'],'full':periods.iloc[0].to_dict(),'illustrative_quarters':key_quarters,'worst126':worst,'best126':best,'years':periods.loc[periods.kind.eq('year')].to_dict(orient='records'),'quarter_count':16,'rolling126_count':819,'daily_component_closure_max':float(np.max(np.abs(d[fields[1:]].sum(axis=1)-d.paired_log_excess))),'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'causal_policy_inference':False,'independent_forward_days':0}
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);dataset.config(dump_all=True,recursive=True)
 rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset)
 rec.save_objects(artifact_path='analysis',daily=d,periods=periods,rolling126=rolling,report=report)
 rec.log_metrics(native_sessions=944,quarter_count=16,rolling126_windows=819,new_model_fits=0,new_native_accounts=0)
summary={'experiment':plan['experiment'],'decision':'descriptive_original_Stage1_signal_cap_loss_chain_no_promotion','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'source_Stage1_recorder_id':src['stage1_recorder_id'],'source_component_recorder_id':src['component_recorder_id'],'new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Account cash and selection contributions are descriptive physical path components, not causal switches.','End-of-interval beta is not exact post-open action beta.','All 2023-2026 history is reused development data; no independent forward evidence.']}
(card/'result.json').write_text(json.dumps(report,indent=2,default=str)+'\n');(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='stage1-loss-chain-restore-') as t:
 archive=Path(t)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/daily') for m in z.getmembers())
summary['nas_evidence']=evidence;summary['nas_readback']={'verified':True,'sha256':evidence['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print({k:report[k] for k in ('status','quarter_count','rolling126_count','daily_component_closure_max')});print({k:report['full'][k] for k in ('relative_wealth','mean_raw_nominal_beta','mean_own_state_beta_cap','cap_binding_fraction')})


2026/10/23 06:45:05 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[8930:MainThread](2026-10-23 06:45:06,003) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[8930:MainThread](2026-10-23 06:45:06,005) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[8930:MainThread](2026-10-23 06:45:06,005) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_domain_heads_20261018')}


AssertionError: 

## 中文结论

# 原 Stage1 0bp 信号至损失诊断 v1：来源日历边界停止

首轮只读 notebook 在 Qlib 日历校验处停止。账户复盘 provider 从 2023-01-04 开始，而冻结原 Stage1 信号帧有 2023-01-03 至 2026-10-07 共 944 日。原 Stage1 39 项所用的 daily_prices provider 包含全部 944 日。未写新 Recorder、未拟合、未跑账户、未重放。v2 只更换诊断 Qlib 日历来源，保留全部源对象、账户损益及分析定义。
